# Member 3: Prompt Engineering, Decoding, and Evaluation

This notebook runs only the work assigned to Member 3:

- **E2:** zero-shot, few-shot, role/context, and constraint-based prompt comparison
- **E3:** greedy, temperature, top-k, and top-p decoding comparison
- **Evaluation:** per-output metrics and an aggregate comparison matrix

It reuses Member 1's prepared dataset and shared model loader so all experiments use the same inputs and model as the baseline.

## 1. Colab runtime setup

Before continuing, select **Runtime > Change runtime type > T4 GPU**. The quantized model loader requires an NVIDIA GPU.

In [2]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/2025paml587-byte/BITS_PAML_AGAI_G10_A1.git"
REPO_DIR = Path("/content/BITS_PAML_AGAI_G10_A1")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(f"Working directory: {Path.cwd()}")

Working directory: /content/BITS_PAML_AGAI_G10_A1


In [3]:
# Runtime packages needed by the shared model loader and Member 3 module.
%pip install -q datasets pandas numpy python-dotenv transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.1 MB/s eta 0:00:00:00:0100:01


In [4]:
import torch

assert torch.cuda.is_available(), (
    "CUDA GPU not detected. In Colab, select Runtime > Change runtime type > T4 GPU."
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")

GPU: Tesla T4
PyTorch: 2.11.0+cu128


## 2. Import and test the Member 3 utilities

This lightweight check validates prompt rendering and evaluation logic before loading the LLM.

In [5]:
from src.member3_experiments import (
    DECODING_CONFIGS,
    PROMPT_VARIANTS,
    build_evaluation_matrix,
    build_prompt,
    load_experiment_samples,
    run_decoding_study,
    run_prompt_comparison,
    save_results,
    set_reproducible_seed,
)

assert len(PROMPT_VARIANTS) == 4
assert len(DECODING_CONFIGS) == 4
test_prompt = build_prompt("zero_shot", "Alex: The meeting is at 3 PM.")
assert "Alex: The meeting is at 3 PM." in test_prompt
print("Member 3 utility checks passed.")
print("Prompt variants:", list(PROMPT_VARIANTS))
print("Decoding configurations:", list(DECODING_CONFIGS))

Member 3 utility checks passed.
Prompt variants: ['zero_shot', 'few_shot', 'role_context', 'constraint_based']
Decoding configurations: ['greedy', 'temperature', 'top_k', 'top_p']


## 3. Load the controlled experiment samples

Three examples satisfy the assignment's minimum evidence requirement. Increase `SAMPLE_LIMIT` after the initial test if more evidence is needed.

In [6]:
SAMPLE_LIMIT = 3
SAMPLING_REPEATS = 3
MAX_NEW_TOKENS = 150
SEED = 42

set_reproducible_seed(SEED)
samples = load_experiment_samples(
    data_path="data/experimental_subset.csv",
    sample_limit=SAMPLE_LIMIT,
)
display(samples[["id", "dialogue", "summary"]])

,id,dialogue,summary
0,13811908,violet hi i came across this austins article a...,violet sent claire austins article
1,13731387,mel right its time i bought some christmas pre...,mel bought christmas presents while evie keeps...
2,13727910,carol can you pls pick up when i call you caro...,fiona cant talk to carol because shes very bus...


## 4. Load the shared quantized model

This uses the repository's shared model configuration rather than introducing a different model for Member 3. The first run downloads the model and may take several minutes.

In [7]:
from src.model_pipeline import load_quantized_model

model, tokenizer = load_quantized_model()
print("Shared model and tokenizer loaded successfully.")

📦 Initializing configuration for model: microsoft/Phi-3-mini-4k-instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

⏳ Loading Tokenizer...


tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

✅ Model and Tokenizer successfully loaded!
Shared model and tokenizer loaded successfully.


## 5. Run E2: four prompt variants

Greedy decoding is fixed for every prompt so the prompt itself is the only experimental variable.

In [8]:
model.generation_config.use_cache = False

In [9]:
import transformers

print("Transformers:", transformers.__version__)
print("Model implementation:", model.__class__.__module__)

Transformers: 5.13.1
Model implementation: transformers.models.phi3.modeling_phi3


In [10]:
e2_records = run_prompt_comparison(
    model=model,
    tokenizer=tokenizer,
    samples=samples,
    max_new_tokens=MAX_NEW_TOKENS,
)
print(f"E2 completed: {len(e2_records)} outputs generated.")

E2 completed: 12 outputs generated.


## 6. Run E3: decoding-control study

The constraint-based prompt is fixed for all runs. Sampling configurations are repeated to measure variability; deterministic greedy decoding runs once per sample.

In [11]:
e3_records = run_decoding_study(
    model=model,
    tokenizer=tokenizer,
    samples=samples,
    prompt_variant="constraint_based",
    max_new_tokens=MAX_NEW_TOKENS,
    sampling_repeats=SAMPLING_REPEATS,
)
print(f"E3 completed: {len(e3_records)} outputs generated.")

E3 completed: 30 outputs generated.


## 7. Build and inspect the evaluation matrix

In [12]:
all_records = e2_records + e3_records
evaluation_details, comparison_matrix = build_evaluation_matrix(all_records)

display(
    comparison_matrix.sort_values(
        ["experiment", "rouge_1_f1"], ascending=[True, False]
    )
)

,experiment,variant,rouge_1_precision,rouge_1_recall,rouge_1_f1,rouge_l_f1,source_token_support,compression_ratio,generated_word_count,latency_seconds,is_empty,output_diversity
3,E2_prompt_comparison,constraint_based,0.1198,0.5791,0.1778,0.1255,0.3207,1.2892,93.6667,60.8337,0.0,1.0
0,E2_prompt_comparison,zero_shot,0.1005,0.5085,0.1533,0.1076,0.2932,1.3290,98.3333,50.0313,0.0,1.0
1,E2_prompt_comparison,few_shot,0.1007,0.4863,0.1502,0.0902,0.3342,1.3534,97.0000,63.5505,0.0,1.0
2,E2_prompt_comparison,role_context,0.0725,0.4405,0.1128,0.0917,0.2958,1.3415,102.6667,59.0495,0.0,1.0
4,E3_decoding_study,greedy,0.1198,0.5791,0.1778,0.1255,0.3207,1.2892,93.6667,60.4783,0.0,1.0
6,E3_decoding_study,top_k,0.1241,0.5068,0.1754,0.1103,0.3435,1.3304,92.4444,55.0194,0.0,1.0
7,E3_decoding_study,top_p,0.1385,0.4488,0.1694,0.1056,0.3180,1.2741,78.6667,43.4484,0.0,1.0
5,E3_decoding_study,temperature,0.1161,0.5124,0.1617,0.1182,0.3093,1.3750,93.0000,53.2186,0.0,1.0


In [13]:
# Report-ready output examples for qualitative comparison.
display(
    evaluation_details[[
        "experiment",
        "variant",
        "trial",
        "sample_id",
        "generated_summary",
        "reference_summary",
        "rouge_1_f1",
        "rouge_l_f1",
        "source_token_support",
    ]].head(20)
)

,experiment,variant,trial,sample_id,generated_summary,reference_summary,rouge_1_f1,rouge_l_f1,source_token_support
0,E2_prompt_comparison,zero_shot,1,13811908,Violet found an article by Austin and shared i...,violet sent claire austins article,0.0606,0.0606,0.1170
1,E2_prompt_comparison,zero_shot,1,13731387,"Mel and Evie discuss their Christmas shopping,...",mel bought christmas presents while evie keeps...,0.2484,0.1491,0.4000
2,E2_prompt_comparison,zero_shot,1,13727910,Carol has been trying to reach Fiona for two w...,fiona cant talk to carol because shes very bus...,0.1509,0.1132,0.3626
3,E2_prompt_comparison,few_shot,1,13811908,Violet found an Austin article and thought Cla...,violet sent claire austins article,0.0588,0.0392,0.1856
4,E2_prompt_comparison,few_shot,1,13731387,Mel and Evie are discussing Christmas preparat...,mel bought christmas presents while evie keeps...,0.2597,0.1558,0.5534
5,E2_prompt_comparison,few_shot,1,13727910,"Carol has been unavailable for two weeks, and ...",fiona cant talk to carol because shes very bus...,0.1321,0.0755,0.2637
6,E2_prompt_comparison,role_context,1,13811908,Claire has already read the article by Austin ...,violet sent claire austins article,0.0652,0.0652,0.1609
7,E2_prompt_comparison,role_context,1,13731387,The conversation revolves around the stress of...,mel bought christmas presents while evie keeps...,0.1646,0.1013,0.4019
8,E2_prompt_comparison,role_context,1,13727910,Carol has been trying to reach Fiona for two w...,fiona cant talk to carol because shes very bus...,0.1085,0.1085,0.3246
9,E2_prompt_comparison,constraint_based,1,13811908,"Violet shared an article with Claire, who had ...",violet sent claire austins article,0.0825,0.0825,0.1522


### Metric interpretation

- **ROUGE-1 / ROUGE-L:** lexical agreement with the reference summary
- **Source-token support:** proportion of generated tokens also present in the dialogue; this is a lexical grounding proxy, not a complete factuality measure
- **Compression ratio:** generated-summary length divided by dialogue length
- **Output diversity:** fraction of unique repeated outputs for the same sample and configuration
- **Latency:** generation time in seconds

Use the displayed examples for qualitative analysis alongside these automated measurements.

## 8. Save and download Member 3 evidence

In [14]:
result_paths = save_results(
    all_records, output_dir="data/member3_results"
)
result_paths

{'outputs': 'data/member3_results/generation_outputs.csv',
 'details': 'data/member3_results/evaluation_details.csv',
 'matrix': 'data/member3_results/comparison_matrix.csv'}

In [16]:
# Optional: download the three CSV evidence files to your computer.
from google.colab import files

for result_path in result_paths.values():
    files.download(result_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>